# Etap 3 – Baza SQL

Tworzenie bazy SQLite, ładowanie danych z `data/processed/clinical_processed.csv` i eksploracja przez zapytania SQL.

**Źródło danych:** `clinical_processed.csv` (1047 pacjentów, 19 kolumn)
**Baza:** `db/tcga_glioma.db`

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

# Ścieżki
DB_PATH = Path('../db/tcga_glioma.db')          # gdzie powstanie plik bazy
CSV_PATH = Path('../data/processed/clinical_processed.csv')
SCHEMA_PATH = Path('../sql/schema.sql')

DB_PATH.parent.mkdir(exist_ok=True)

print("Ścieżki OK")
print(f"  Baza:   {DB_PATH}")
print(f"  Dane:   {CSV_PATH}")
print(f"  Schema: {SCHEMA_PATH}")

Ścieżki OK
  Baza:   ..\db\tcga_glioma.db
  Dane:   ..\data\processed\clinical_processed.csv
  Schema: ..\sql\schema.sql


## 1. Tworzenie bazy i ładowanie schematu

In [2]:
with open(SCHEMA_PATH, 'r') as f:
    schema_sql = f.read()

conn = sqlite3.connect(DB_PATH)

conn.executescript(schema_sql)
conn.commit()

print("Baza utworzona, tabele gotowe")

Baza utworzona, tabele gotowe


In [3]:
# Wyczyść tabele przed ponownym ładowaniem
conn.executescript("""
    DELETE FROM biomarkers;
    DELETE FROM survival;
    DELETE FROM patients;
""")
conn.commit()
print("Tabele wyczyszczone")

Tabele wyczyszczone


## 2. Ładowanie danych do tabel

In [4]:

df = pd.read_csv(CSV_PATH)


df_patients = df[['patient_id', 'study', 'histology', 'grade',
                   'age_at_diagnosis', 'gender', 'kps', 'mutation_count']]

df_biomarkers = df[['patient_id', 'idh_status', 'mgmt_status', 'codel_1p19q',
                     'idh_codel_subtype', 'tert_promoter_status', 'atrx_status',
                     'idh_mutant', 'mgmt_methylated', 'codel']]

df_survival = df[['patient_id', 'os_months', 'os_event']]


df_patients.to_sql('patients', conn, if_exists='append', index=False)
df_biomarkers.to_sql('biomarkers', conn, if_exists='append', index=False)
df_survival.to_sql('survival', conn, if_exists='append', index=False)

conn.commit()

print(f"Załadowano pacjentów:  {len(df_patients)}")
print(f"Załadowano biomarkerów: {len(df_biomarkers)}")
print(f"Załadowano survival:    {len(df_survival)}")

Załadowano pacjentów:  1047
Załadowano biomarkerów: 1047
Załadowano survival:    1047


## 3. Weryfikacja załadowanych danych

In [5]:
# Sprawdź liczbę wierszy w każdej tabeli
for tabela in ['patients', 'biomarkers', 'survival']:
    wynik = pd.read_sql(f"SELECT COUNT(*) AS liczba_wierszy FROM {tabela}", conn)
    print(f"{tabela}: {wynik['liczba_wierszy'][0]} wierszy")

patients: 1047 wierszy
biomarkers: 1047 wierszy
survival: 1047 wierszy


In [8]:
# 1. Ile pacjentów mamy w każdym typie guza?
query1 = """
SELECT study, COUNT(*) AS liczba_pacjentow
FROM patients
GROUP BY study;
"""
pd.read_sql_query(query1, conn)

,study,liczba_pacjentow
0,GBM,590
1,LGG,457


In [9]:
# 2. Pacjenci powyżej 60 lat, tylko GBM, posortowani od najstarszych
query2 = """
SELECT patient_id, age_at_diagnosis, gender
FROM patients
WHERE study = 'GBM' AND age_at_diagnosis > 60
ORDER BY age_at_diagnosis DESC;
"""
pd.read_sql_query(query2, conn)

,patient_id,age_at_diagnosis,gender
0,TCGA-41-2571,89.0,male
1,TCGA-41-3392,88.0,male
2,TCGA-06-0126,86.0,male
3,TCGA-06-0140,86.0,male
4,TCGA-12-1600,86.0,male
...,...,...,...
261,TCGA-12-3648,61.0,female
262,TCGA-14-1043,61.0,male
263,TCGA-14-1455,61.0,male
264,TCGA-19-0963,61.0,male


In [10]:
# 3. Jakie unikalne typy histologiczne występują w danych?
query3 = """
SELECT DISTINCT histology
FROM patients;
"""
pd.read_sql_query(query3, conn)

,histology
0,astrocytoma
1,oligodendroglioma
2,oligoastrocytoma
3,glioblastoma


In [11]:
# 4. Średni wiek diagnozy per typ guza
query4 = """
SELECT study, AVG(age_at_diagnosis) AS sredni_wiek, COUNT(*) AS n
FROM patients
GROUP BY study;
"""
pd.read_sql_query(query4, conn)

,study,sredni_wiek,n
0,GBM,57.822034,590
1,LGG,43.161926,457


In [12]:
# 5. Typy histologiczne, które mają więcej niż 50 pacjentów
query5 = """
SELECT histology, COUNT(*) AS n
FROM patients
GROUP BY histology
HAVING COUNT(*) > 50;
"""
pd.read_sql_query(query5, conn)

,histology,n
0,astrocytoma,169
1,glioblastoma,590
2,oligoastrocytoma,114
3,oligodendroglioma,174


### 4. Zapytania eksploracyjne

In [13]:
# 6. Wiek i płeć pacjentów z IDH mutant
query6 = """
SELECT p.patient_id, p.age_at_diagnosis, p.gender, b.idh_status
FROM patients p
JOIN biomarkers b ON p.patient_id = b.patient_id
WHERE b.idh_status = 'Mutant'
ORDER BY p.age_at_diagnosis;
"""
pd.read_sql_query(query6, conn)

,patient_id,age_at_diagnosis,gender,idh_status
0,TCGA-HT-7483,14.0,male,Mutant
1,TCGA-DB-5278,17.0,male,Mutant
2,TCGA-HT-7482,18.0,female,Mutant
3,TCGA-TM-A84T,19.0,male,Mutant
4,TCGA-DB-A64S,20.0,male,Mutant
...,...,...,...,...
401,TCGA-S9-A6WH,73.0,female,Mutant
402,TCGA-DU-A6S8,74.0,female,Mutant
403,TCGA-FG-A713,74.0,female,Mutant
404,TCGA-HT-7687,74.0,male,Mutant


In [15]:
# 7. Pełny obraz pacjenta: dane kliniczne + biomarkery + przeżycie
query7 = """
SELECT p.patient_id, p.study, p.age_at_diagnosis,
       b.idh_status, b.mgmt_status,
       s.os_months, s.os_event
FROM patients p
JOIN biomarkers b ON p.patient_id = b.patient_id
JOIN survival s ON p.patient_id = s.patient_id
LIMIT 10;
"""
pd.read_sql_query(query7, conn)

,patient_id,study,age_at_diagnosis,idh_status,mgmt_status,os_months,os_event
0,TCGA-CS-4938,LGG,31.0,Mutant,Unmethylated,4.698251,0.0
1,TCGA-CS-4941,LGG,67.0,WT,Methylated,7.688047,1.0
2,TCGA-CS-4942,LGG,44.0,Mutant,Unmethylated,43.861292,1.0
3,TCGA-CS-4943,LGG,37.0,Mutant,Methylated,18.135905,0.0
4,TCGA-CS-4944,LGG,50.0,Mutant,Methylated,10.612133,0.0
5,TCGA-CS-5390,LGG,47.0,Mutant,Methylated,64.592733,0.0
6,TCGA-CS-5393,LGG,39.0,Mutant,Methylated,40.148688,0.0
7,TCGA-CS-5394,LGG,40.0,Mutant,Methylated,0.098565,0.0
8,TCGA-CS-5395,LGG,43.0,WT,Unmethylated,20.994281,1.0
9,TCGA-CS-5396,LGG,53.0,Mutant,Methylated,9.955035,0.0


In [16]:
# 8. Czy są pacjenci bez danych o przeżyciu? (LEFT JOIN sprawdza braki)
query8 = """
SELECT p.patient_id, s.os_months, s.os_event
FROM patients p
LEFT JOIN survival s ON p.patient_id = s.patient_id
WHERE s.os_months IS NULL;
"""
pd.read_sql_query(query8, conn)

,patient_id,os_months,os_event


In [18]:
# 9. Średni i medialny OS wg statusu IDH
query9 = """
SELECT b.idh_status,
       AVG(s.os_months) AS sredni_os,
       COUNT(*) AS n
FROM biomarkers b
JOIN survival s ON b.patient_id = s.patient_id
GROUP BY b.idh_status;
"""
pd.read_sql_query(query9, conn)

,idh_status,sredni_os,n
0,NaN,15.697039,121
1,Mutant,27.085385,406
2,WT,13.721217,520


In [19]:
# 10. To samo, ale osobno dla MGMT
query10 = """
SELECT b.mgmt_status,
       AVG(s.os_months) AS sredni_os,
       COUNT(*) AS n
FROM biomarkers b
JOIN survival s ON b.patient_id = s.patient_id
GROUP BY b.mgmt_status;
"""
pd.read_sql_query(query10, conn)

,mgmt_status,sredni_os,n
0,NaN,17.622836,185
1,Methylated,21.915098,560
2,Unmethylated,14.895237,302


In [20]:
# 11. Współwystępowanie IDH i MGMT, cross-tab przez GROUP BY na dwóch kolumnach naraz
query11 = """
SELECT idh_status, mgmt_status, COUNT(*) AS n
FROM biomarkers
GROUP BY idh_status, mgmt_status
ORDER BY idh_status, mgmt_status;
"""
pd.read_sql_query(query11, conn)

,idh_status,mgmt_status,n
0,NaN,NaN,71
1,NaN,Methylated,26
2,NaN,Unmethylated,24
3,Mutant,NaN,7
4,Mutant,Methylated,367
5,Mutant,Unmethylated,32
6,WT,NaN,107
7,WT,Methylated,167
8,WT,Unmethylated,246


In [21]:
# 12. Ile brakuje danych o biomarkerach, CTE do policzenia i zaraportowania % braków
query12 = """
WITH braki AS (
    SELECT
        SUM(CASE WHEN idh_status IS NULL THEN 1 ELSE 0 END) AS brak_idh,
        SUM(CASE WHEN mgmt_status IS NULL THEN 1 ELSE 0 END) AS brak_mgmt,
        COUNT(*) AS total
    FROM biomarkers
)
SELECT total,
       brak_idh,
       ROUND(100.0 * brak_idh / total, 1) AS pct_brak_idh,
       brak_mgmt,
       ROUND(100.0 * brak_mgmt / total, 1) AS pct_brak_mgmt
FROM braki;
"""
pd.read_sql_query(query12, conn)

,total,brak_idh,pct_brak_idh,brak_mgmt,pct_brak_mgmt
0,1047,121,11.6,185,17.7


Pacjenci z brakiem statusu IDH i/lub MGMT (235/1047, 22,4%) są wykluczani z analiz wykorzystujących te zmiennei. Efektywna kohorta do modelu Coxa (Etap 5): n=812.

In [22]:
# 13. Pacjenci starsi niż średnia wieku w całej kohorcie
query13 = """
SELECT patient_id, study, age_at_diagnosis
FROM patients
WHERE age_at_diagnosis > (SELECT AVG(age_at_diagnosis) FROM patients)
ORDER BY age_at_diagnosis DESC
LIMIT 10;
"""
pd.read_sql_query(query13, conn)

,patient_id,study,age_at_diagnosis
0,TCGA-41-2571,GBM,89.0
1,TCGA-41-3392,GBM,88.0
2,TCGA-DU-A76K,LGG,87.0
3,TCGA-06-0126,GBM,86.0
4,TCGA-06-0140,GBM,86.0
5,TCGA-12-1600,GBM,86.0
6,TCGA-16-0846,GBM,85.0
7,TCGA-76-4928,GBM,85.0
8,TCGA-06-0122,GBM,84.0
9,TCGA-06-2559,GBM,83.0


In [23]:
# 14. Ranking pacjentów wg wieku wewnątrz każdego typu guza
query14 = """
SELECT patient_id, study, age_at_diagnosis,
       RANK() OVER (PARTITION BY study ORDER BY age_at_diagnosis DESC) AS ranking_wieku
FROM patients
ORDER BY study, ranking_wieku
LIMIT 20;
"""
pd.read_sql_query(query14, conn)

,patient_id,study,age_at_diagnosis,ranking_wieku
0,TCGA-41-2571,GBM,89.0,1
1,TCGA-41-3392,GBM,88.0,2
2,TCGA-06-0126,GBM,86.0,3
3,TCGA-06-0140,GBM,86.0,3
4,TCGA-12-1600,GBM,86.0,3
5,TCGA-16-0846,GBM,85.0,6
6,TCGA-76-4928,GBM,85.0,6
7,TCGA-06-0122,GBM,84.0,8
8,TCGA-06-2559,GBM,83.0,9
9,TCGA-19-0960,GBM,83.0,9
